# Waze IA: rutas en La Estrella

Este cuaderno aplica A* propio sobre la red vial de La Estrella, Antioquia. Reutiliza el flujo de localización, tablas geográficas y visualización trabajado en clase, pero conserva el modelo de costos por tipo de vía, intersecciones y giros.


## 1. Importaciones y mapa

La red se descarga como un grafo dirigido para vehículos. NetworkX solo se usa para representar y consultar el grafo: la búsqueda de rutas se implementa más adelante con A*.


In [ ]:
import os
from pathlib import Path
import heapq
import math
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
import pandas as pd
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from shapely.geometry import LineString

pd.set_option("display.max_columns", None)


In [ ]:
G = ox.graph_from_place('La Estrella, Antioquia, Colombia', network_type='drive', simplify=False)
ox.plot_graph(G, figsize=(25, 25))

## 2. Información del mapa y tiempos de viaje

Se convierten los nodos y aristas a tablas geográficas y se calculan velocidades y tiempos de viaje, igual que en el ejemplo de clase.


In [ ]:
hwy_speeds = {
    'motorway': 80,
    'trunk': 60,
    'primary': 50,
    'secondary': 40,
    'tertiary': 35,
    'residential': 30,
    'unclassified': 30,
    'service': 20
}

hwy_speeds

In [ ]:
G = ox.add_edge_speeds(G, hwy_speeds=hwy_speeds)
G = ox.add_edge_travel_times(G)
G = ox.add_edge_bearings(G)

print('Velocidades, tiempos de viaje y orientaciones calculados correctamente.')

In [ ]:
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges.head()

In [ ]:
gdf_edges.explode('highway').groupby('highway')[['length', 'speed_kph', 'travel_time']].mean().round(1)

## 3. Origen y destino

Se localizan Pitriza y La Tablaza. Las coordenadas verificadas funcionan como respaldo si Nominatim no devuelve un resultado.


In [ ]:
locator = Nominatim(user_agent="waze_ia_la_estrella")


In [ ]:
coordenadas_respaldo = {
    'Pitriza': (6.1562385, -75.6362846),
    'La Tablaza': (6.1181940, -75.6351392),
}


def geocodificar_o_usar_respaldo(query):
    try:
        return locator.geocode(query)
    except Exception as error:
        print(f'No se pudo consultar Nominatim: {error}. Se usar? el respaldo.')
        return None


location_start = geocodificar_o_usar_respaldo('Pitriza, La Estrella, Antioquia, Colombia')
location_end = geocodificar_o_usar_respaldo(
    'Coliseo Y Unidad Deportiva La Tablaza, La Estrella, Antioquia, Colombia'
)


In [ ]:
print('Origen encontrado:' if location_start else 'Origen: usando respaldo manual')
if location_start:
    print(location_start.address)
    print((location_start.latitude, location_start.longitude))
else:
    print(coordenadas_respaldo['Pitriza'])

print('\nDestino encontrado:' if location_end else '\nDestino: usando respaldo manual')
if location_end:
    print(location_end.address)
    print((location_end.latitude, location_end.longitude))
else:
    print(coordenadas_respaldo['La Tablaza'])

In [ ]:
start = (location_start.latitude, location_start.longitude) if location_start else coordenadas_respaldo['Pitriza']
end = (location_end.latitude, location_end.longitude) if location_end else coordenadas_respaldo['La Tablaza']

print('Coordenadas del origen:', start)
print('Coordenadas del destino:', end)

In [ ]:
start_node = ox.distance.nearest_nodes(G, start[1], start[0])
end_node = ox.distance.nearest_nodes(G, end[1], end[0])

print('Nodo vial del origen:', start_node)
print('Nodo vial del destino:', end_node)

In [ ]:
fig, ax = ox.plot_graph(G, figsize=(25, 25), node_size=0, show=False, close=False)

ax.scatter(G.nodes[start_node]['x'], G.nodes[start_node]['y'],
           c='red', s=250, label='Nodo vial Pitriza', zorder=3)
ax.scatter(G.nodes[end_node]['x'], G.nodes[end_node]['y'],
           c='green', s=250, label='Nodo vial La Tablaza', zorder=3)
ax.scatter(start[1], start[0], c='darkred', s=180, marker='s',
           label='Coordenada Pitriza', zorder=4)
ax.scatter(end[1], end[0], c='darkgreen', s=180, marker='s',
           label='Coordenada La Tablaza', zorder=4)

ax.legend(fontsize=14)
plt.show()

## 4. A* propio

Cada estado representa un nodo de la red vial. La implementación respeta vías dirigidas y aristas paralelas, y no usa algoritmos de rutas de NetworkX.


In [ ]:
class OSMRouteMap:
    """
    Consultas, costos y heurística para una red vial dirigida
    obtenida de OpenStreetMap mediante OSMnx.

    La función objetivo es minimizar el tiempo estimado de viaje
    en automóvil.
    """

    def __init__(self, graph, config=None):
        self.graph = graph
        self.config = dict(config) if config is not None else {}

        # La velocidad de referencia nunca puede ser menor que la
        # velocidad máxima real del grafo, porque la heurística
        # podría dejar de ser admisible.
        graph_max_speed_kph = self._maximum_speed_kph()

        if 'max_speed_kph' in self.config:
            self.max_speed_kph = float(self.config['max_speed_kph'])
        else:
            self.max_speed_kph = graph_max_speed_kph

        if (
            not math.isfinite(self.max_speed_kph)
            or self.max_speed_kph <= 0
        ):
            raise ValueError(
                'max_speed_kph debe ser un número positivo.'
            )

        if self.max_speed_kph < graph_max_speed_kph:
            raise ValueError(
                'max_speed_kph debe ser igual o superior a la '
                'velocidad máxima del grafo '
                f'({graph_max_speed_kph:.1f} km/h).'
            )

    @staticmethod
    def _as_numbers(value):
        """
        Convierte valores simples o listas de OSM en números válidos.

        OSM puede almacenar algunos atributos como:
            30
            '30'
            ['30', '40']
        """
        values = (
            value
            if isinstance(value, (list, tuple))
            else [value]
        )

        numbers = []

        for item in values:
            try:
                number = float(item)
            except (TypeError, ValueError):
                continue

            if math.isfinite(number):
                numbers.append(number)

        return numbers

    def _maximum_speed_kph(self):
        """
        Obtiene la mayor velocidad válida presente en las aristas.

        Esta velocidad se utiliza como referencia para construir
        la heurística temporal.
        """
        speeds = []

        for _, _, edge_data in self.graph.edges(data=True):
            speeds.extend(
                self._as_numbers(
                    edge_data.get('speed_kph')
                )
            )

        if not speeds:
            raise ValueError(
                'El grafo no contiene valores válidos de speed_kph.'
            )

        return max(speeds)

    def neighbors(self, node_id):
        """
        Devuelve las aristas salientes de node_id.

        Cada elemento tiene la forma:

            (neighbor_id, key, edge_data)

        Se utilizan out_edges() y keys=True para:
        - respetar el sentido de circulación;
        - conservar las aristas paralelas.
        """
        return [
            (neighbor_id, key, edge_data)
            for _, neighbor_id, key, edge_data
            in self.graph.out_edges(
                node_id,
                keys=True,
                data=True
            )
        ]

    def edge_travel_cost(
        self,
        u,
        v,
        key,
        edge_data
    ):
        """
        Devuelve el costo de recorrer una arista en segundos.

        El costo corresponde directamente a travel_time,
        calculado previamente por OSMnx.
        """
        travel_times = self._as_numbers(
            edge_data.get('travel_time')
        )

        if not travel_times:
            raise ValueError(
                f'La arista ({u}, {v}, {key}) '
                'no tiene un travel_time válido.'
            )

        travel_time = travel_times[0]

        if travel_time < 0:
            raise ValueError(
                f'La arista ({u}, {v}, {key}) '
                'tiene un costo negativo.'
            )

        return travel_time

    def heuristic(self, node_id, destination_id):
        """
        Estima en segundos el tiempo mínimo restante desde
        node_id hasta destination_id.

        La estimación se obtiene mediante:

            distancia en línea recta / velocidad máxima

        La distancia se calcula mediante la fórmula de Haversine.

        Utilizar la distancia en línea recta proporciona una
        distancia mínima posible entre ambos nodos. Utilizar la
        velocidad máxima proporciona el menor tiempo físicamente
        posible bajo el modelo utilizado.

        Por tanto, la heurística no sobreestima el tiempo real
        de viaje y es admisible para A* bajo este modelo.
        """
        if node_id == destination_id:
            return 0.0

        origin = self.graph.nodes[node_id]
        destination = self.graph.nodes[destination_id]

        # Coordenadas geográficas de los nodos.
        lat1 = math.radians(origin['y'])
        lon1 = math.radians(origin['x'])

        lat2 = math.radians(destination['y'])
        lon2 = math.radians(destination['x'])

        # Diferencias angulares.
        delta_lat = lat2 - lat1
        delta_lon = lon2 - lon1

        # Fórmula de Haversine.
        a = (
            math.sin(delta_lat / 2) ** 2
            + math.cos(lat1)
            * math.cos(lat2)
            * math.sin(delta_lon / 2) ** 2
        )

        # Evita pequeños errores numéricos fuera del intervalo [0, 1].
        a = min(1.0, max(0.0, a))

        earth_radius_m = 6_371_008.8

        distance_m = (
            2
            * earth_radius_m
            * math.asin(math.sqrt(a))
        )

        # Conversión:
        #
        # distancia_m / (km/h)
        # ->
        # segundos
        #
        # 1 km/h = 1/3.6 m/s
        estimated_time_s = (
            distance_m * 3.6 / self.max_speed_kph
        )

        return estimated_time_s

    def a_star(self, start_id, destination_id):
        """
        Encuentra una ruta de costo temporal mínimo mediante A*.

        La frontera utiliza heapq y ordena los estados por:

            f(n) = g(n) + h(n)

        El resultado contiene la ruta, su costo y métricas de la
        búsqueda. Si no existe una ruta, found es False.
        """
        if start_id not in self.graph:
            raise ValueError(
                f'El nodo de origen {start_id} no existe en el grafo.'
            )

        if destination_id not in self.graph:
            raise ValueError(
                f'El nodo de destino {destination_id} no existe en el grafo.'
            )

        started_at = time.perf_counter()

        # Cada entrada contiene:
        # (f_score, g_al_insertar, contador, node_id)
        # El contador evita comparar identificadores en caso de empate.
        frontier = []
        push_counter = 0

        g_score = {start_id: 0.0}
        came_from = {}
        explored_nodes = set()

        initial_f = self.heuristic(start_id, destination_id)
        heapq.heappush(
            frontier,
            (initial_f, 0.0, push_counter, start_id)
        )

        while frontier:
            _, queued_g, _, current = heapq.heappop(frontier)

            # Una mejora posterior puede dejar entradas antiguas
            # dentro del heap. Esas entradas se descartan aquí.
            if queued_g > g_score.get(current, math.inf):
                continue

            explored_nodes.add(current)

            if current == destination_id:
                route_nodes, route_edges = self.reconstruct_route(
                    came_from,
                    destination_id
                )

                return {
                    'found': True,
                    'route_nodes': route_nodes,
                    'route_edges': route_edges,
                    'total_cost_s': g_score[destination_id],
                    'explored_nodes': len(explored_nodes),
                    'elapsed_time_s': time.perf_counter() - started_at,
                    'message': 'Ruta encontrada.'
                }

            for neighbor, key, edge_data in self.neighbors(current):
                step_cost = self.edge_travel_cost(
                    current,
                    neighbor,
                    key,
                    edge_data
                )
                tentative_g = queued_g + step_cost

                if tentative_g < g_score.get(neighbor, math.inf):
                    g_score[neighbor] = tentative_g
                    came_from[neighbor] = (current, key)

                    push_counter += 1
                    estimated_f = (
                        tentative_g
                        + self.heuristic(neighbor, destination_id)
                    )
                    heapq.heappush(
                        frontier,
                        (
                            estimated_f,
                            tentative_g,
                            push_counter,
                            neighbor
                        )
                    )

        return {
            'found': False,
            'route_nodes': [],
            'route_edges': [],
            'total_cost_s': math.inf,
            'explored_nodes': len(explored_nodes),
            'elapsed_time_s': time.perf_counter() - started_at,
            'message': (
                f'No existe una ruta dirigida entre {start_id} '
                f'y {destination_id}.'
            )
        }

    def reconstruct_route(
        self,
        came_from,
        destination_id
    ):
        """
        Reconstruye los nodos y las aristas desde el destino.

        came_from guarda para cada nodo:

            node_id -> (predecessor_id, edge_key)
        """
        route_nodes = [destination_id]
        route_edges = []
        current = destination_id

        while current in came_from:
            predecessor, key = came_from[current]
            route_edges.append((predecessor, current, key))
            current = predecessor
            route_nodes.append(current)

        route_nodes.reverse()
        route_edges.reverse()

        return route_nodes, route_edges

In [ ]:
route_map = OSMRouteMap(G)
resultado_astar = route_map.a_star(start_node, end_node)

print(resultado_astar['message'])
print('Nodo de origen:', start_node)
print('Nodo de destino:', end_node)

if resultado_astar['found']:
    print('Inicio de la ruta:', resultado_astar['route_nodes'][0])
    print('Final de la ruta:', resultado_astar['route_nodes'][-1])
    print('Cantidad de nodos de la ruta:', len(resultado_astar['route_nodes']))
    print('Cantidad de aristas de la ruta:', len(resultado_astar['route_edges']))
    print('Costo temporal (s):', round(resultado_astar['total_cost_s'], 2))
    print('Costo temporal (min):', round(resultado_astar['total_cost_s'] / 60, 2))

print('Nodos explorados:', resultado_astar['explored_nodes'])
print('Duración de la búsqueda (s):', round(resultado_astar['elapsed_time_s'], 6))

## 5. Validación y costos compuestos

Primero se verifica A* contra costo uniforme propio. Después se ejecutan los cuatro perfiles de costo: tiempo base, tipo de vía, intersecciones y giros.


In [ ]:
route_nodes = resultado_astar['route_nodes']
route_edges = resultado_astar['route_edges']

empieza_en_origen = (
    resultado_astar['found']
    and route_nodes[0] == start_node
)
termina_en_destino = (
    resultado_astar['found']
    and route_nodes[-1] == end_node
)
cantidad_consistente = (
    len(route_nodes) == len(route_edges) + 1
)

aristas_dirigidas_validas = all(
    u == route_nodes[index]
    and v == route_nodes[index + 1]
    and G.has_edge(u, v, key)
    for index, (u, v, key) in enumerate(route_edges)
)

costos_tramos = [
    route_map.edge_travel_cost(
        u,
        v,
        key,
        G.edges[u, v, key]
    )
    for u, v, key in route_edges
]
costos_validos = all(
    math.isfinite(costo) and costo >= 0
    for costo in costos_tramos
)
costo_recalculado_s = sum(costos_tramos)
costo_coincide = math.isclose(
    costo_recalculado_s,
    resultado_astar['total_cost_s'],
    rel_tol=1e-12,
    abs_tol=1e-9
)
heuristica_destino_cero = math.isclose(
    route_map.heuristic(end_node, end_node),
    0.0,
    abs_tol=1e-12
)

validaciones_astar = pd.DataFrame({
    'validación': [
        'Ruta encontrada',
        'Empieza en el origen',
        'Termina en el destino',
        'Cantidad nodos/aristas consistente',
        'Aristas dirigidas y claves válidas',
        'Costos finitos y no negativos',
        'Suma de costos coincide',
        'h(destino) = 0',
    ],
    'cumple': [
        resultado_astar['found'],
        empieza_en_origen,
        termina_en_destino,
        cantidad_consistente,
        aristas_dirigidas_validas,
        costos_validos,
        costo_coincide,
        heuristica_destino_cero,
    ]
})

assert validaciones_astar['cumple'].all(), (
    'La ruta A* no superó todas las validaciones.'
)

print('Costo reportado (s):', resultado_astar['total_cost_s'])
print('Costo recalculado (s):', costo_recalculado_s)
validaciones_astar

In [ ]:
class UniformCostRouteMap(OSMRouteMap):
    def heuristic(self, node_id, destination_id):
        return 0.0


uniform_cost_map = UniformCostRouteMap(G)
resultado_costo_uniforme = uniform_cost_map.a_star(
    start_node,
    end_node
)

costos_iguales = (
    resultado_costo_uniforme['found']
    and math.isclose(
        resultado_astar['total_cost_s'],
        resultado_costo_uniforme['total_cost_s'],
        rel_tol=1e-12,
        abs_tol=1e-9
    )
)

assert resultado_costo_uniforme['found'], (
    'La búsqueda de costo uniforme no encontró una ruta.'
)
assert costos_iguales, (
    'A* y costo uniforme encontraron costos diferentes.'
)

comparacion_busquedas = pd.DataFrame([
    {
        'algoritmo': 'A*',
        'costo total (s)': resultado_astar['total_cost_s'],
        'nodos explorados': resultado_astar['explored_nodes'],
        'duración (s)': resultado_astar['elapsed_time_s'],
    },
    {
        'algoritmo': 'Costo uniforme propio',
        'costo total (s)': resultado_costo_uniforme['total_cost_s'],
        'nodos explorados': resultado_costo_uniforme['explored_nodes'],
        'duración (s)': resultado_costo_uniforme['elapsed_time_s'],
    },
])

comparacion_busquedas['costo total (s)'] = (
    comparacion_busquedas['costo total (s)'].round(6)
)
comparacion_busquedas['duración (s)'] = (
    comparacion_busquedas['duración (s)'].round(6)
)

print('Los costos encontrados son iguales:', costos_iguales)
comparacion_busquedas

In [ ]:
class CompositeOSMRouteMap(OSMRouteMap):
    """A* propio con costo temporal compuesto y estado ampliado."""

    DEFAULT_CONFIG = {
        'highway_speeds_kph': {
            'motorway': 80.0,
            'trunk': 60.0,
            'primary': 50.0,
            'secondary': 40.0,
            'tertiary': 35.0,
            'residential': 30.0,
            'unclassified': 30.0,
            'service': 20.0,
            'trunk_link': 40.0,
            'primary_link': 40.0,
            'secondary_link': 30.0,
            'tertiary_link': 25.0,
        },
        'default_speed_kph': 30.0,
        'highway_time_factors': {
            'motorway': 1.00,
            'trunk': 1.00,
            'primary': 1.00,
            'secondary': 1.02,
            'tertiary': 1.05,
            'residential': 1.10,
            'unclassified': 1.08,
            'service': 1.15,
            'trunk_link': 1.02,
            'primary_link': 1.02,
            'secondary_link': 1.04,
            'tertiary_link': 1.06,
        },
        'default_highway_time_factor': 1.05,
        'intersection_delay_s_by_street_count': {
            3: 2.0,
            4: 4.0,
            5: 6.0,
        },
        'straight_threshold_deg': 30.0,
        'u_turn_threshold_deg': 150.0,
        'turn_delay_s': {
            'recto': 0.0,
            'giro': 3.0,
            'retorno': 12.0,
        },
    }

    def __init__(self, graph, config=None):
        merged = dict(self.DEFAULT_CONFIG)
        if config is not None:
            merged.update(config)
        self._validate_cost_config(merged)
        super().__init__(graph, config=merged)

    @staticmethod
    def _as_osm_values(value):
        if isinstance(value, (list, tuple, set)):
            return list(value)
        return [value]

    @staticmethod
    def _validate_cost_config(config):
        delays = list(config['intersection_delay_s_by_street_count'].values())
        delays += list(config['turn_delay_s'].values())
        speeds = list(config['highway_speeds_kph'].values())
        speeds.append(config['default_speed_kph'])
        factors = list(config['highway_time_factors'].values())
        factors.append(config['default_highway_time_factor'])

        if any(not math.isfinite(float(value)) or float(value) < 0 for value in delays):
            raise ValueError('Todas las demoras deben ser finitas y no negativas.')
        if any(not math.isfinite(float(value)) or float(value) <= 0 for value in speeds):
            raise ValueError('Todas las velocidades deben ser finitas y positivas.')
        if any(not math.isfinite(float(value)) or float(value) < 1 for value in factors):
            raise ValueError('Los factores de tipo de vía deben ser finitos y mayores o iguales a 1.')
        if not (0 <= config['straight_threshold_deg'] < config['u_turn_threshold_deg'] <= 180):
            raise ValueError('Los umbrales angulares deben cumplir 0 <= recto < retorno <= 180.')

    def highway_speed_kph(self, edge_data):
        """Usa speed_kph y, si falta, lo estima mediante highway."""
        speeds = self._as_numbers(edge_data.get('speed_kph'))
        speeds = [speed for speed in speeds if speed > 0]
        if speeds:
            return speeds[0]

        estimated = [
            float(self.config['highway_speeds_kph'].get(
                str(highway),
                self.config['default_speed_kph']
            ))
            for highway in self._as_osm_values(edge_data.get('highway'))
        ]
        return min(estimated, default=float(self.config['default_speed_kph']))

    def highway_time_factor(self, edge_data):
        """Factor temporal adicional asociado a la jerarquía highway."""
        factors = [
            float(self.config['highway_time_factors'].get(
                str(highway),
                self.config['default_highway_time_factor']
            ))
            for highway in self._as_osm_values(edge_data.get('highway'))
        ]
        return max(
            factors,
            default=float(self.config['default_highway_time_factor'])
        )

    def intersection_delay(self, node_id, destination_id=None):
        """Demora al atravesar una intersección según street_count."""
        if node_id == destination_id:
            return 0.0

        try:
            street_count = int(self.graph.nodes[node_id].get('street_count', 0))
        except (TypeError, ValueError):
            return 0.0

        applicable = [
            int(threshold)
            for threshold in self.config['intersection_delay_s_by_street_count']
            if street_count >= int(threshold)
        ]
        if not applicable:
            return 0.0
        threshold = max(applicable)
        return float(self.config['intersection_delay_s_by_street_count'][threshold])

    @staticmethod
    def angular_difference(bearing_in, bearing_out):
        """Menor diferencia entre dos orientaciones, entre 0 y 180 grados."""
        return abs((bearing_out - bearing_in + 180.0) % 360.0 - 180.0)

    def turn_delay(self, node_id, incoming_data, outgoing_data):
        """Clasifica el movimiento y devuelve su demora en segundos."""
        if incoming_data is None:
            return 0.0, 'inicio', None
        if self.graph.nodes[node_id].get('street_count', 0) < 3:
            return 0.0, 'continuidad', None

        incoming = self._as_numbers(incoming_data.get('bearing'))
        outgoing = self._as_numbers(outgoing_data.get('bearing'))
        if not incoming or not outgoing:
            return 0.0, 'sin_dato', None

        angle = self.angular_difference(incoming[0], outgoing[0])
        if angle < float(self.config['straight_threshold_deg']):
            movement = 'recto'
        elif angle <= float(self.config['u_turn_threshold_deg']):
            movement = 'giro'
        else:
            movement = 'retorno'

        delay = float(self.config['turn_delay_s'].get(movement, 0.0))
        return delay, movement, angle

    def edge_travel_cost(
        self,
        state,
        u,
        v,
        key,
        edge_data,
        destination_id=None
    ):
        """Calcula tiempo + tipo de vía + intersección + giro en segundos."""
        speed_kph = self.highway_speed_kph(edge_data)
        travel_times = self._as_numbers(edge_data.get('travel_time'))

        if travel_times:
            base_time = travel_times[0]
        else:
            lengths = self._as_numbers(edge_data.get('length'))
            if not lengths:
                raise ValueError(
                    f'La arista ({u}, {v}, {key}) no tiene length ni travel_time válidos.'
                )
            base_time = lengths[0] * 3.6 / speed_kph

        if not math.isfinite(base_time) or base_time < 0:
            raise ValueError(f'Tiempo base inválido en ({u}, {v}, {key}).')

        highway_factor = self.highway_time_factor(edge_data)
        highway_adjustment = base_time * (highway_factor - 1.0)
        intersection_adjustment = self.intersection_delay(v, destination_id)

        previous, _, incoming_key = state
        incoming_data = None
        if previous is not None:
            incoming_data = self.graph.edges[previous, u, incoming_key]

        turn_adjustment, movement, angle = self.turn_delay(
            u, incoming_data, edge_data
        )
        total = (
            base_time
            + highway_adjustment
            + intersection_adjustment
            + turn_adjustment
        )

        if not math.isfinite(total) or total < 0:
            raise ValueError(f'Costo total inválido en ({u}, {v}, {key}).')

        return total, {
            'base_time_s': base_time,
            'highway_adjustment_s': highway_adjustment,
            'highway_time_factor': highway_factor,
            'intersection_adjustment_s': intersection_adjustment,
            'turn_adjustment_s': turn_adjustment,
            'total_edge_cost_s': total,
            'movement': movement,
            'turn_angle_deg': angle,
            'street_count_arrival': self.graph.nodes[v].get('street_count', 0),
            'highway': ', '.join(
                map(str, self._as_osm_values(edge_data.get('highway')))
            ),
            'speed_kph': speed_kph,
            'length_m': float(edge_data.get('length', 0.0)),
        }

    def a_star(self, start_id, destination_id):
        """A* usando estados (anterior, actual, clave de llegada)."""
        if start_id not in self.graph or destination_id not in self.graph:
            raise ValueError('El origen y el destino deben existir en el grafo.')

        started_at = time.perf_counter()
        start_state = (None, start_id, None)
        frontier = []
        push_counter = 0
        g_score = {start_state: 0.0}
        came_from = {}
        explored_states = set()

        heapq.heappush(
            frontier,
            (self.heuristic(start_id, destination_id), 0.0, push_counter, start_state)
        )

        while frontier:
            _, queued_g, _, state = heapq.heappop(frontier)
            if queued_g > g_score.get(state, math.inf):
                continue

            previous, current, incoming_key = state
            explored_states.add(state)

            if current == destination_id:
                route_nodes, route_edges, details = self.reconstruct_composite_route(
                    came_from, state
                )
                return {
                    'found': True,
                    'route_nodes': route_nodes,
                    'route_edges': route_edges,
                    'edge_details': details,
                    'total_cost_s': queued_g,
                    'explored_nodes': len({item[1] for item in explored_states}),
                    'explored_states': len(explored_states),
                    'elapsed_time_s': time.perf_counter() - started_at,
                    'message': 'Ruta de costo compuesto encontrada.'
                }

            for neighbor, key, edge_data in self.neighbors(current):
                step_cost, detail = self.edge_travel_cost(
                    state, current, neighbor, key, edge_data, destination_id
                )
                next_state = (current, neighbor, key)
                tentative_g = queued_g + step_cost

                if tentative_g < g_score.get(next_state, math.inf):
                    g_score[next_state] = tentative_g
                    came_from[next_state] = (
                        state, (current, neighbor, key), detail
                    )
                    push_counter += 1
                    estimated_f = (
                        tentative_g
                        + self.heuristic(neighbor, destination_id)
                    )
                    heapq.heappush(
                        frontier,
                        (estimated_f, tentative_g, push_counter, next_state)
                    )

        return {
            'found': False,
            'route_nodes': [],
            'route_edges': [],
            'edge_details': [],
            'total_cost_s': math.inf,
            'explored_nodes': len({item[1] for item in explored_states}),
            'explored_states': len(explored_states),
            'elapsed_time_s': time.perf_counter() - started_at,
            'message': f'No existe una ruta dirigida entre {start_id} y {destination_id}.'
        }

    @staticmethod
    def reconstruct_composite_route(came_from, destination_state):
        route_nodes = [destination_state[1]]
        route_edges = []
        details = []
        state = destination_state

        while state in came_from:
            predecessor_state, edge, detail = came_from[state]
            route_edges.append(edge)
            details.append(detail)
            state = predecessor_state
            route_nodes.append(state[1])

        route_nodes.reverse()
        route_edges.reverse()
        details.reverse()
        return route_nodes, route_edges, details

In [ ]:
perfiles_costo = {
    '1. Solo travel_time': {
        'highway_time_factors': {},
        'default_highway_time_factor': 1.0,
        'intersection_delay_s_by_street_count': {},
        'turn_delay_s': {'recto': 0.0, 'giro': 0.0, 'retorno': 0.0},
    },
    '2. Travel_time + tipo de vía': {
        'intersection_delay_s_by_street_count': {},
        'turn_delay_s': {'recto': 0.0, 'giro': 0.0, 'retorno': 0.0},
    },
    '3. Travel_time + tipo de vía + intersecciones': {
        'turn_delay_s': {'recto': 0.0, 'giro': 0.0, 'retorno': 0.0},
    },
    '4. Modelo completo': {},
}

resultados_compuestos = {}
for nombre, configuracion in perfiles_costo.items():
    model = CompositeOSMRouteMap(G, configuracion)
    result = model.a_star(start_node, end_node)
    assert result['found'], f'No se encontró una ruta para {nombre}.'
    resultados_compuestos[nombre] = result

print('Configuraciones ejecutadas:')
for nombre in resultados_compuestos:
    print('-', nombre)

In [ ]:
for nombre, result in resultados_compuestos.items():
    nodes = result['route_nodes']
    edges = result['route_edges']
    details = result['edge_details']

    assert nodes[0] == start_node and nodes[-1] == end_node
    assert len(nodes) == len(edges) + 1 == len(details) + 1
    assert all(
        u == nodes[index]
        and v == nodes[index + 1]
        and G.has_edge(u, v, key)
        for index, (u, v, key) in enumerate(edges)
    )

    recalculated = sum(detail['total_edge_cost_s'] for detail in details)
    assert all(detail['total_edge_cost_s'] >= 0 for detail in details)
    assert math.isclose(
        recalculated, result['total_cost_s'], rel_tol=1e-10, abs_tol=1e-7
    )

# Sin penalizaciones, el estado ampliado debe conservar el óptimo de la fase 6.
assert math.isclose(
    resultados_compuestos['1. Solo travel_time']['total_cost_s'],
    resultado_astar['total_cost_s'],
    rel_tol=1e-10,
    abs_tol=1e-7
)

class CompositeUniformCostRouteMap(CompositeOSMRouteMap):
    def heuristic(self, node_id, destination_id):
        return 0.0

full_astar = resultados_compuestos['4. Modelo completo']
full_uniform = CompositeUniformCostRouteMap(G).a_star(start_node, end_node)

assert full_uniform['found']
assert math.isclose(
    full_astar['total_cost_s'],
    full_uniform['total_cost_s'],
    rel_tol=1e-10,
    abs_tol=1e-7
)

print('Todas las rutas y sus costos son válidos.')
print('A* compuesto y costo uniforme encontraron el mismo costo.')

In [ ]:
comparacion_costos = []
for nombre, result in resultados_compuestos.items():
    details = pd.DataFrame(result['edge_details'])
    comparacion_costos.append({
        'configuración': nombre,
        'distancia (km)': details['length_m'].sum() / 1000,
        'tiempo base (min)': details['base_time_s'].sum() / 60,
        'ajuste tipo de vía (s)': details['highway_adjustment_s'].sum(),
        'demora intersecciones (s)': details['intersection_adjustment_s'].sum(),
        'demora giros (s)': details['turn_adjustment_s'].sum(),
        'costo ajustado (min)': result['total_cost_s'] / 60,
        'intersecciones': int((details['intersection_adjustment_s'] > 0).sum()),
        'giros': int((details['movement'] == 'giro').sum()),
        'retornos': int((details['movement'] == 'retorno').sum()),
        'estados explorados': result['explored_states'],
        'duración (s)': result['elapsed_time_s'],
    })

comparacion_costos = pd.DataFrame(comparacion_costos)
columnas_decimales = [
    'distancia (km)',
    'tiempo base (min)',
    'ajuste tipo de vía (s)',
    'demora intersecciones (s)',
    'demora giros (s)',
    'costo ajustado (min)',
    'duración (s)',
]
comparacion_costos[columnas_decimales] = (
    comparacion_costos[columnas_decimales].round(3)
)
comparacion_costos

In [ ]:
resultado_final = resultados_compuestos['4. Modelo completo']
detalles_finales = pd.DataFrame(resultado_final['edge_details'])

detalles_finales.insert(0, 'arista', resultado_final['route_edges'])
detalles_finales.insert(1, 'desde', [u for u, _, _ in resultado_final['route_edges']])
detalles_finales.insert(2, 'hasta', [v for _, v, _ in resultado_final['route_edges']])

componentes_costo = [
    'base_time_s',
    'highway_adjustment_s',
    'intersection_adjustment_s',
    'turn_adjustment_s',
]

costo_recalculado_s = detalles_finales[componentes_costo].sum().sum()
assert math.isclose(
    costo_recalculado_s,
    resultado_final['total_cost_s'],
    rel_tol=1e-10,
    abs_tol=1e-7
)

print('Ruta final preparada correctamente.')

In [ ]:
resumen_ruta_final = pd.Series({
    'Distancia total (km)': detalles_finales['length_m'].sum() / 1000,
    'Travel time o tiempo base (min)': detalles_finales['base_time_s'].sum() / 60,
    'Ajuste por tipo de vía (s)': detalles_finales['highway_adjustment_s'].sum(),
    'Demora por intersecciones (s)': detalles_finales['intersection_adjustment_s'].sum(),
    'Demora por giros (s)': detalles_finales['turn_adjustment_s'].sum(),
    'TIEMPO TOTAL ESTIMADO (min)': resultado_final['total_cost_s'] / 60,
    'Intersecciones penalizadas': int(
        (detalles_finales['intersection_adjustment_s'] > 0).sum()
    ),
    'Giros': int((detalles_finales['movement'] == 'giro').sum()),
    'Retornos': int((detalles_finales['movement'] == 'retorno').sum()),
    'Nodos en la ruta': len(resultado_final['route_nodes']),
    'Estados explorados por A*': resultado_final['explored_states'],
    'Duración de la búsqueda (s)': resultado_final['elapsed_time_s'],
}).round(3)

display(resumen_ruta_final.to_frame(name='valor'))

## 6. Tabla geográfica de la ruta final

A diferencia de conectar solo los nodos, esta tabla conserva la geometría exacta de cada arista seleccionada por A*.


In [ ]:
# Se conserva la arista exacta (u, v, key) elegida por A* y su geometría.
def geometry_of_edge(u, v, key):
    edge_data = G.edges[u, v, key]
    geometry = edge_data.get("geometry")
    if geometry is not None:
        return geometry
    return LineString([
        (G.nodes[u]["x"], G.nodes[u]["y"]),
        (G.nodes[v]["x"], G.nodes[v]["y"]),
    ])


tramos_geograficos = []
for order, ((u, v, key), detail) in enumerate(
    zip(resultado_final["route_edges"], resultado_final["edge_details"]), start=1
):
    geometry = geometry_of_edge(u, v, key)
    tramos_geograficos.append({
        "orden": order,
        "node_start": u,
        "node_end": v,
        "edge_key": key,
        "length_m": detail["length_m"],
        "travel_time_s": detail["base_time_s"],
        "highway": detail["highway"],
        "speed_kph": detail["speed_kph"],
        "movement": detail["movement"],
        "total_edge_cost_s": detail["total_edge_cost_s"],
        "geometry": geometry,
    })

gdf_ruta = gpd.GeoDataFrame(
    tramos_geograficos, geometry="geometry", crs="EPSG:4326"
)
df_ruta = gdf_ruta.drop(columns="geometry").copy()

print("Tramos de la ruta:", len(gdf_ruta))
display(gdf_ruta.head())


## 7. Visualizaciones de la ruta

Primero se presenta la vista estática; luego el mapa interactivo y la animación.


In [ ]:
fig, ax = ox.plot_graph(
    G,
    figsize=(16, 16),
    node_size=0,
    edge_color='#C8C8C8',
    edge_linewidth=0.5,
    show=False,
    close=False
)

for u, v, key in resultado_final['route_edges']:
    edge_data = G.edges[u, v, key]
    geometry = edge_data.get('geometry')

    if geometry is None:
        x = [G.nodes[u]['x'], G.nodes[v]['x']]
        y = [G.nodes[u]['y'], G.nodes[v]['y']]
    else:
        x, y = geometry.xy

    ax.plot(x, y, color='#1565C0', linewidth=3, zorder=3)

ax.scatter(
    G.nodes[start_node]['x'], G.nodes[start_node]['y'],
    color='red', s=120, label='Pitriza', zorder=4
)
ax.scatter(
    G.nodes[end_node]['x'], G.nodes[end_node]['y'],
    color='green', s=120, label='La Tablaza', zorder=4
)
ax.set_title(
    f"Ruta A* con costo compuesto: {resultado_final['total_cost_s'] / 60:.2f} min",
    fontsize=15
)
ax.legend(fontsize=12)
plt.show()

### Mapa interactivo: Mapbox o MapLibre

El notebook lee el token desde el archivo local `.env`, que est? ignorado por Git. Copia `.env.example` como `.env` y cambia ?nicamente el valor de `MAPBOX_TOKEN`. Si el archivo no existe o no contiene un token, el notebook utiliza MapLibre sin token. La rama Mapbox conserva `Scattermapbox` por compatibilidad con el material de clase, aunque Plotly la considera una API obsoleta.


In [ ]:
def load_mapbox_token(env_path='.env'):
    """Lee MAPBOX_TOKEN desde .env sin imprimir ni exponer su valor."""
    path = Path(env_path)
    if not path.is_file():
        return None

    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == 'MAPBOX_TOKEN':
            return value.strip().strip('"').strip("'") or None
    return None


# Copia .env.example como .env y cambia solo MAPBOX_TOKEN.
# El archivo .env est? ignorado por Git y el token nunca se muestra en pantalla.
MAPBOX_TOKEN = load_mapbox_token()
if MAPBOX_TOKEN and not MAPBOX_TOKEN.startswith('pk.'):
    raise ValueError('MAPBOX_TOKEN debe ser un token p?blico de Mapbox que comience por pk.')
print('Modo de mapa:', 'Mapbox desde .env' if MAPBOX_TOKEN else 'MapLibre sin token')


def route_coordinates(route_gdf):
    """Devuelve coordenadas lon/lat separadas por None para cada tramo."""
    lon, lat = [], []
    for geometry in route_gdf.geometry:
        coordinates = list(geometry.coords)
        lon.extend(point[0] for point in coordinates)
        lat.extend(point[1] for point in coordinates)
        lon.append(None)
        lat.append(None)
    return lon, lat


def route_center(route_gdf):
    min_x, min_y, max_x, max_y = route_gdf.total_bounds
    return {"lon": (min_x + max_x) / 2, "lat": (min_y + max_y) / 2}


def make_route_map(route_gdf, start_node, end_node, token=None):
    """Crea un mapa Mapbox si hay token; de lo contrario usa MapLibre."""
    lon, lat = route_coordinates(route_gdf)
    center = route_center(route_gdf)
    start_lon, start_lat = G.nodes[start_node]["x"], G.nodes[start_node]["y"]
    end_lon, end_lat = G.nodes[end_node]["x"], G.nodes[end_node]["y"]

    if token:
        # Scattermapbox se conserva para compatibilidad con el cuaderno docente.
        # Plotly lo marca como obsoleto; el modo sin token utiliza MapLibre.
        figure = go.Figure([
            go.Scattermapbox(lon=lon, lat=lat, mode="lines", name="Ruta A*",
                             line={"width": 5, "color": "#1565C0"}),
            go.Scattermapbox(lon=[start_lon], lat=[start_lat], mode="markers",
                             name="Pitriza", marker={"size": 14, "color": "#D32F2F"}),
            go.Scattermapbox(lon=[end_lon], lat=[end_lat], mode="markers",
                             name="La Tablaza", marker={"size": 14, "color": "#2E7D32"}),
        ])
        figure.update_layout(mapbox={
            "accesstoken": token,
            "style": "streets",
            "center": center,
            "zoom": 13,
        })
        provider = "Mapbox (streets)"
    else:
        figure = go.Figure([
            go.Scattermap(lon=lon, lat=lat, mode="lines", name="Ruta A*",
                          line={"width": 5, "color": "#1565C0"}),
            go.Scattermap(lon=[start_lon], lat=[start_lat], mode="markers",
                          name="Pitriza", marker={"size": 14, "color": "#D32F2F"}),
            go.Scattermap(lon=[end_lon], lat=[end_lat], mode="markers",
                          name="La Tablaza", marker={"size": 14, "color": "#2E7D32"}),
        ])
        figure.update_layout(map={"style": "carto-positron", "center": center, "zoom": 13})
        provider = "MapLibre (carto-positron, sin token)"

    figure.update_layout(
        title=f"Ruta final A* — {provider}", height=650,
        margin={"r": 10, "t": 50, "l": 10, "b": 10},
        legend={"orientation": "h", "y": 0.02, "x": 0.01},
    )
    return figure


mapa_interactivo = make_route_map(gdf_ruta, start_node, end_node, MAPBOX_TOKEN)
mapa_interactivo.show()


### Animación por tramos

La línea azul avanza sobre la geometría real de las aristas de la ruta final.


In [ ]:
def make_route_animation(route_gdf, start_node, end_node, token=None):
    """Anima el avance acumulado por cada tramo de la ruta final."""
    full_lon, full_lat = route_coordinates(route_gdf)
    center = route_center(route_gdf)
    start_lon, start_lat = G.nodes[start_node]["x"], G.nodes[start_node]["y"]
    end_lon, end_lat = G.nodes[end_node]["x"], G.nodes[end_node]["y"]
    trace_class = go.Scattermapbox if token else go.Scattermap

    frames, current_lon, current_lat = [], [], []
    for row in route_gdf.itertuples():
        coordinates = list(row.geometry.coords)
        current_lon.extend(point[0] for point in coordinates)
        current_lat.extend(point[1] for point in coordinates)
        frames.append(go.Frame(
            name=str(row.orden),
            data=[trace_class(lon=current_lon.copy(), lat=current_lat.copy(), mode="lines",
                              line={"width": 5, "color": "#1565C0"})],
            traces=[0],
        ))

    figure = go.Figure([
        trace_class(lon=full_lon, lat=full_lat, mode="lines", name="Recorrido",
                    line={"width": 5, "color": "rgba(21,101,192,0.20)"}),
        trace_class(lon=[start_lon], lat=[start_lat], mode="markers", name="Pitriza",
                    marker={"size": 14, "color": "#D32F2F"}),
        trace_class(lon=[end_lon], lat=[end_lat], mode="markers", name="La Tablaza",
                    marker={"size": 14, "color": "#2E7D32"}),
    ], frames=frames)
    if token:
        figure.update_layout(mapbox={"accesstoken": token,
                                     "style": "streets",
                                     "center": center, "zoom": 13})
    else:
        figure.update_layout(map={"style": "carto-positron", "center": center, "zoom": 13})

    figure.update_layout(
        title="Animación de la ruta final por tramos", height=650,
        margin={"r": 10, "t": 50, "l": 10, "b": 10},
        updatemenus=[{"type": "buttons", "showactive": False, "x": 0.05, "y": 0.02,
                      "buttons": [{"label": "Reproducir", "method": "animate",
                                   "args": [None, {"frame": {"duration": 180, "redraw": True},
                                                   "fromcurrent": True}]}]}],
        sliders=[{"active": 0, "x": 0.16, "y": 0.02, "len": 0.78,
                  "steps": [{"label": frame.name, "method": "animate",
                             "args": [[frame.name], {"mode": "immediate",
                                                     "frame": {"duration": 0, "redraw": True}}]}
                            for frame in frames]}],
    )
    return figure


animacion_ruta = make_route_animation(gdf_ruta, start_node, end_node, MAPBOX_TOKEN)
animacion_ruta.show()


## Conclusión

El resultado mantiene el A* implementado para el proyecto, explica sus costos y presenta la ruta con la estructura de análisis geográfico usada en clase.
